# Data Engineering Oxford Hands Dataset

In [13]:
def convert_mat_to_yolo(mat_path, image_width, image_height):
    # Load .mat file
    mat_data = scipy.io.loadmat(mat_path)
    
    yolo_annotations = []
    for box in mat_data['boxes'][0]:  # First [0] because boxes is a 1D array
        # Each box is a structured array with fields a, b, c, d
        # Extract the coordinates from each point
        points = []
        for point_field in ['a', 'b', 'c', 'd']:
            # Get both x and y coordinates for each point
            y = box[point_field][0][0][0][0].item()  # x coordinate
            x = box[point_field][0][0][0][1].item()  # y coordinate
            points.append([x, y])

        # Convert points to numpy array for easier manipulation
        points = np.array(points)
        p = str(points)
        
        # Calculate bounding box from the four points
        x_min = np.min(points[:, 0])
        y_min = np.min(points[:, 1])
        x_max = np.max(points[:, 0])
        y_max = np.max(points[:, 1])
        
        # Convert to YOLO format (x_center, y_center, width, height)
        x_center = (x_min + x_max) / (2 * image_width)
        y_center = (y_min + y_max) / (2 * image_height)
        width = (x_max - x_min) / image_width
        height = (y_max - y_min) / image_height
        
        # Ensure values are within [0, 1]
        x_center = max(0, min(1, x_center))
        y_center = max(0, min(1, y_center))
        width = max(0, min(1, width))
        height = max(0, min(1, height))
        
        # Class id is 0 for hands (we're not distinguishing between left and right hands)
        yolo_annotations.append(f"0 {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")
        # yolo_annotations.append(p)
    
    return yolo_annotations

def process_dataset(base_path, subset):
    mat_dir = os.path.join(base_path, subset, 'labels')
    img_dir = os.path.join(base_path, subset, 'images')
    
    for mat_file in os.listdir(mat_dir):
        if mat_file.endswith('.mat'):
            base_name = mat_file[:-4]
            mat_path = os.path.join(mat_dir, mat_file)
            
            # Get corresponding image dimensions
            img_path = os.path.join(img_dir, f"{base_name}.jpg")
            img = cv2.imread(img_path)
            if img is None:
                print(f"Error: Image {img_path} not found")
                continue
            height, width = img.shape[:2]
            
            # Convert annotations
            yolo_annotations = convert_mat_to_yolo(mat_path, width, height)
            
            # Save as .txt file
            txt_path = os.path.join(mat_dir, f"{base_name}.txt")
            try:
                with open(txt_path, 'w') as f:
                    f.write('\n'.join(yolo_annotations))
            except Exception as e:
                print(f"Error saving file: {e}")                        
            
            # Remove the .mat file (optional)
            os.remove(mat_path)

# Process all subsets
# dataset_path = oxford dataset
# for subset in ['train', 'val', 'test']:
#     process_dataset(dataset_path, subset)

# Data Engineering Egohands Dataset 

In [ ]:
def segmentation_to_box(seg):
    """
    Given a polygon segmentation (an array of Nx2 points),
    compute the bounding box as [x, y, width, height].
    """
    
    x_min = np.min(seg[:, 0])
    y_min = np.min(seg[:, 1])
    x_max = np.max(seg[:, 0])
    y_max = np.max(seg[:, 1])
    w = x_max - x_min
    h = y_max - y_min
    return [x_min, y_min, w, h]

def process_egohands(metadata_path, base_image_path, output_label_path, output_image_path):
    """
    Process the EgoHands metadata to extract bounding boxes.
    
    Args:
      metadata_path (str): Path to metadata.mat.
      base_image_path (str): Directory containing EgoHands videos, e.g. '_LABELLED_SAMPLES'.
      output_label_path (str): Directory where the YOLO annotation TXT files will be stored.
    """
    # create output directories if they don't exist
    os.makedirs(output_label_path, exist_ok=True)
    os.makedirs(output_image_path, exist_ok=True)
    
    data = scipy.io.loadmat(metadata_path, squeeze_me=True, struct_as_record=False)
    videos = data['video']
    
    # Ensure videos is iterable even if there's only one
    videos = np.atleast_1d(videos)
    
    # Create the output directory for annotations if it doesn't exist
    # os.makedirs(output_label_path, exist_ok=True)
    
    for vid in videos:
        video_id = vid.video_id  # e.g., "JENGA_LIVINGROOM_B_S"
        video_folder = os.path.join(base_image_path, video_id)
        labelled_frames = np.atleast_1d(vid.labelled_frames)
        
        for frame in labelled_frames:
            frame_num = frame.frame_num  # The frame number (e.g., 1, 2, …)
            # Construct the image filename using the typical EgoHands naming convention
            img_filename = f"frame_{int(frame_num):04d}.jpg"
            img_path = os.path.join(video_folder, img_filename)
            
            # Check if the image exists
            if not os.path.exists(img_path):
                print(f"Image not found: {img_path}")
                continue
                        
            # Read image (optional: to verify dimensions)
            img = cv2.imread(img_path)
            if img is None:
                print(f"Could not open image: {img_path}")
                continue
            else:
                img_height, img_width = img.shape[:2]
            
            # Collect bounding boxes from the segmentation fields
            boxes = []
            for field in ['myleft', 'myright', 'yourleft', 'yourright']:
                seg = getattr(frame, field, None)
                if seg is None:
                    continue
                if seg.size == 0:
                    continue
                # Ensure seg is a numpy array (it might be a list)
                seg = np.array(seg)
                # Sometimes the segmentation polygon is 2D: shape (N,2)
                # If it's not, try to reshape
                if seg.ndim != 2 or seg.shape[1] != 2:
                    try:
                        seg = seg.reshape(-1, 2)
                    except Exception as e:
                        print(f"Could not reshape segmentation for {video_id} frame {frame_num} field {field}")
                        continue
                box = segmentation_to_box(seg)
                if box is not None:
                    boxes.append(box)
            
            if not boxes:
                continue
            
            # Convert each absolute bounding box to YOLO normalized format.
            yolo_annotations = []
            for box in boxes:
                x, y, w, h = box
                center_x = (x + w/2) / img_width
                center_y = (y + h/2) / img_height
                norm_w = w / img_width
                norm_h = h / img_height
                # Here, class id is assumed to be 0 for "hand"
                yolo_annotations.append(f"0 {center_x:.6f} {center_y:.6f} {norm_w:.6f} {norm_h:.6f}")
            
            # Create a label file for this frame.
            # Here we name the label file with videoID and frame number to ensure uniqueness.
            label_filename = f"{video_id}_frame_{int(frame_num):04d}.txt"
            label_file_path = os.path.join(output_label_path, label_filename)
            with open(label_file_path, "w") as f:
                f.write("\n".join(yolo_annotations))
            
            # Copy the image to the output image path with a filename matching the associated label file
            img_filename = f"{video_id}_frame_{int(frame_num):04d}.jpg"
            output_img_path = os.path.join(output_image_path, img_filename)
            shutil.copy(img_path, output_img_path)

# usage:
metadata_path = "./ego(copy)/metadata.mat"
base_image_path = "./ego(copy)/_LABELLED_SAMPLES"
output_label_path = "./ego(copy)/labels"
output_image_path = "./ego(copy)/images"

process_egohands(metadata_path, base_image_path, output_label_path, output_image_path)

In [ ]:
def yolo_directory_structure(base_path, output_path, train_ratio=0.8, val_ratio=0.1):
    """
    Uses or creates a YOLO-style directory structure and splits the dataset.
    If output_path already exists, it will use the existing directories.
    If it doesn't exist, it will create the necessary directory structure.
    """
    # First check if output_path exists
    if not os.path.exists(output_path):
        os.makedirs(output_path)
        print(f"Created new directory: {output_path}")
    else:
        print(f"Using existing directory: {output_path}")

    # Get or create the necessary subdirectories
    train_path = os.path.join(output_path, "train")
    val_path = os.path.join(output_path, "val")
    test_path = os.path.join(output_path, "test")
    
    # For each split directory, ensure images and labels subdirectories exist
    for split_path in [train_path, val_path, test_path]:
        if not os.path.exists(split_path):
            os.makedirs(os.path.join(split_path, "images"))
            os.makedirs(os.path.join(split_path, "labels"))
            print(f"Created directory structure in: {split_path}")
    
    # Define paths for later use
    train_images_path = os.path.join(train_path, "images")
    train_labels_path = os.path.join(train_path, "labels")
    val_images_path = os.path.join(val_path, "images")
    val_labels_path = os.path.join(val_path, "labels")
    test_images_path = os.path.join(test_path, "images")
    test_labels_path = os.path.join(test_path, "labels")

    # Original images and labels folders
    orig_images_path = os.path.join(base_path, "images")
    orig_labels_path = os.path.join(base_path, "labels")

    # Get list of all image files
    images = os.listdir(orig_images_path)
    images = [img for img in images if img.lower().endswith(('.jpg', '.jpeg', '.png'))]
    
    # Shuffle the image list
    np.random.shuffle(images)
    num_images = len(images)

    # Calculate the split indices based on the ratios
    num_train = int(num_images * train_ratio)
    num_val = int(num_images * val_ratio)
    num_test = num_images - num_train - num_val

    train_files = images[:num_train]
    val_files = images[num_train:num_train+num_val]
    test_files = images[num_train+num_val:]
    
    # Helper function to copy files along with their corresponding label file
    def copy_files(file_list, dst_images_path, dst_labels_path):
        for file in file_list:
            src_image = os.path.join(orig_images_path, file)
            src_label = os.path.join(orig_labels_path, os.path.splitext(file)[0] + ".txt")
            dst_image = os.path.join(dst_images_path, file)
            dst_label = os.path.join(dst_labels_path, os.path.splitext(file)[0] + ".txt")
            shutil.copy(src_image, dst_image)
            if os.path.exists(src_label):
                shutil.copy(src_label, dst_label)
            else:
                print(f"Warning: Label file {src_label} does not exist.")
    
    # Copy files into the structure
    copy_files(train_files, train_images_path, train_labels_path)
    copy_files(val_files, val_images_path, val_labels_path)
    copy_files(test_files, test_images_path, test_labels_path)

    print(f"Dataset split: {len(train_files)} training, {len(val_files)} validation, and {len(test_files)} testing images.")

# Example usage:
base_path = "./ego(copy)"
output_path = "./datasets"  # Will use existing directory if it exists
yolo_directory_structure(base_path, output_path)

# YOLO Models Validation

In [42]:
# # Create data.yaml file
# yaml_content = f"""
# path: ./datasets     # dataset root dir
# train: train/images  # train images (relative to 'path')
# val: val/images      # val images (relative to 'path')
# test: test/images    # test images (relative to 'path')

# # Classes
# names:
#   0: hand  # class names
# """

# with open('./datasets/data.yaml', 'w') as f:
#     f.write(yaml_content)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

# # Load models
# model_8s_pretrained = YOLO('yolov8s.pt')
# model_11s_pretrained = YOLO('yolo11s.pt')
model_11n_pretrained = YOLO('yolo11n.pt')
# model_8s_finetuned = YOLO('./runs/detect/hand_detection_8s/weights/best.pt')
# model_11s_finetuned = YOLO('./runs/detect/hand_detection_11s/weights/best.pt')
model_11n_finetuned = YOLO('./runs/detect/hand_detection_11n/weights/best.pt')

# # Test the pretrained model on your test set
# results_8s_pretrained = model_8s_pretrained.val(data='./datasets/data.yaml', split='test', name='pretrained_8s_val')
# results_11s_pretrained = model_11s_pretrained.val(data='./datasets/data.yaml', split='test', name='pretrained_11s_val')
results_11n_pretrained = model_11n_pretrained.val(data='./datasets/data.yaml', split='test', name='pretrained_11n_val')

# # Test the fine-tuned model
# results_8s_finetuned = model_8s_finetuned.val(data='./datasets/data.yaml', split='test', name='finetuned_8s_val')
# results_11s_finetuned = model_11s_finetuned.val(data='./datasets/data.yaml', split='test', name='finetuned_11s_val')
results_11n_finetuned = model_11n_finetuned.val(data='./datasets/data.yaml', split='test', name='finetuned_11n_val')

# def plot_results(pretrained_metrics, finetuned_metrics, metric_name):
#     plt.figure(figsize=(10, 6))
    
#     # Map our metric names to readable labels
#     metric_mapping = {
#         'map50': 'mAP50',
#         'map75': 'mAP75',
#         'map50-95': 'mAP50-95',
#         'precision': 'Precision',
#         'recall': 'Recall'
#     }
    
#     # Extract values from the 'box' Metric property
#     if metric_name == 'precision':
#          pretrained_value = pretrained_metrics.box.mp  # Mean precision property
#          finetuned_value = finetuned_metrics.box.mp
#     elif metric_name == 'recall':
#          pretrained_value = pretrained_metrics.box.mr  # Mean recall property
#          finetuned_value = finetuned_metrics.box.mr
#     elif metric_name == 'map50':
#          pretrained_value = pretrained_metrics.box.map50  # mAP50 property
#          finetuned_value = finetuned_metrics.box.map50
#     elif metric_name == 'map75':
#          pretrained_value = pretrained_metrics.box.map75  # mAP75 property
#          finetuned_value = finetuned_metrics.box.map75
#     elif metric_name == 'map50-95':
#          pretrained_value = pretrained_metrics.box.map  # mAP50-95 property
#          finetuned_value = finetuned_metrics.box.map
#     else:
#          raise ValueError(f"Unknown metric: {metric_name}")
    
#     plt.bar(['Pretrained', 'Fine-tuned'], 
#             [pretrained_value, finetuned_value],
#             color=['blue', 'green'])
#     plt.title(f'Comparison of {metric_mapping[metric_name]}')
#     plt.ylabel(metric_mapping[metric_name])
#     plt.ylim(0, 1)  # assuming values are normalized between 0 and 1
#     plt.show()

# # Plot various metrics for both model sizes
# metrics_to_plot = ['map50', 'map75', 'map50-95', 'precision', 'recall']
# for metric in metrics_to_plot:
#     plot_results(results_8s_pretrained, results_8s_finetuned, metric)
#     plot_results(results_11s_pretrained, results_11s_finetuned, metric)
    
# # Save the fine-tuned models to ONNX format
# model_8s_finetuned.export(format='onnx')
# model_11s_finetuned.export(format='onnx')
model_11n_finetuned.export(format='onnx')


# Annotated Image visualization

In [3]:
def visualize_annotations(image_path):
    # Read image
    img = cv2.imread(image_path)
    if img is None:
        print(f"Error: Could not read image {image_path}")
        return
    
    # Convert BGR to RGB for matplotlib
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    height, width = img.shape[:2]
    
    # Get corresponding annotation file path
    base_name = os.path.splitext(image_path)[0]
    txt_path = base_name.replace('images', 'labels') + '.txt'

    plt.figure(figsize=(12, 8))
    plt.imshow(img)

    try:
        with open(txt_path, 'r') as f:
            annotations = f.readlines()

        # Draw each box
        for ann in annotations:
            # Parse YOLO format: class x_center y_center width height
            class_id, x_center, y_center, width_norm, height_norm = map(float, ann.strip().split())
            
            # Convert normalized coordinates to pixel coordinates
            x_center = int(x_center * width)
            y_center = int(y_center * height)
            box_width = int(width_norm * width)
            box_height = int(height_norm * height)
            
            # Calculate corners
            x1 = x_center - box_width/2
            y1 = y_center - box_height/2
            x2 = x_center + box_width/2
            y2 = y_center + box_height/2
            
            # Draw rectangle
            rect = plt.Rectangle((x1, y1), box_width, box_height, 
                               fill=False, color='g', linewidth=2)
            plt.gca().add_patch(rect)
            
            # Draw center point
            plt.plot(x_center, y_center, 'r.', markersize=10)
            
    except Exception as e:
        print(f"Error reading or processing annotation file: {e}")
        return

    plt.axis('off')
    plt.show()

In [ ]:
from IPython.display import clear_output

dir_path = './datasets/val/images'
files = os.listdir(dir_path)
np.random.shuffle(files)

for f in files:
    clear_output(wait=True)
    visualize_annotations(dir_path + '/' + f)
    
    key = input("Press 'n' for next image, 'q' to quit: ")
    plt.close()  # Close the current window
    
    if key.lower() == 'q':
        break

# Analysis

In [ ]:
# For YOLOv8s
log_file = './runs/detect/hand_detection_8s/results.csv'
import pandas as pd
history_8s = pd.read_csv(log_file)
print(f"YOLOv8s trained for {len(history_8s)} epochs")

# For YOLO11s
log_file = './runs/detect/hand_detection_11s/results.csv'
history_11s = pd.read_csv(log_file)
print(f"YOLO11s trained for {len(history_11s)} epochs")

In [ ]:
# The plots are saved in the runs directory
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# For YOLOv8s
plot_8s = mpimg.imread('./runs/detect/hand_detection_8s/results.png')
plt.figure(figsize=(15, 10))
plt.imshow(plot_8s)
plt.axis('off')
plt.title('YOLOv8s Training History')
plt.show()

# For YOLO11s
plot_11s = mpimg.imread('./runs/detect/hand_detection_11s/results.png')
plt.figure(figsize=(15, 10))
plt.imshow(plot_11s)
plt.axis('off')
plt.title('YOLO11s Training History')
plt.show()

In [3]:
# Load model metadata
model_8s = YOLO('./runs/detect/hand_detection_8s/weights/best.pt')
model_11s = YOLO('./runs/detect/hand_detection_11s/weights/best.pt')
model_11n = YOLO('./runs/detect/hand_detection_11n/weights/best.pt')

# Print training info
print("YOLOv8s training info:", model_8s.info())
print("YOLO11s training info:", model_11s.info())
print("YOLO11n training info:", model_11n.info())

Model summary: 129 layers, 11,135,987 parameters, 0 gradients, 28.6 GFLOPs
YOLOv8s training info: (129, 11135987, 0, 28.6469632)
YOLO11s summary: 181 layers, 9,428,179 parameters, 0 gradients, 21.5 GFLOPs
YOLO11s training info: (181, 9428179, 0, 21.548492800000002)
YOLO11n summary: 181 layers, 2,590,035 parameters, 0 gradients, 6.4 GFLOPs
YOLO11n training info: (181, 2590035, 0, 6.4406016)


# Testing

In [1]:
import scipy.io
import os
import numpy as np
import cv2
from ultralytics import YOLO
import matplotlib.pyplot as plt
import shutil
import time

In [2]:
import time
import os
import cv2

# First, close any existing camera connections
# os.system('v4l2-ctl --device=/dev/video3 --all')  # Just to check current settings
# os.system('v4l2-ctl --device=/dev/video3 --set-fmt-video=width=640,height=480,pixelformat=YUYV')

# Now open the camera
cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)

# Verify the settings
actual_width = cap.get(cv2.CAP_PROP_FRAME_WIDTH)
actual_height = cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
print(f"Camera initialized with resolution: {actual_width}x{actual_height}")

cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
print(f"Camera initialized with resolution: {640}x{640}")

prev_time = time.time()

while True:
    ret, frame = cap.read()
    if not ret:
        print("Failed to grab frame")
        break
    
    # frame = cv2.flip(frame, 1)
    
    current_time = time.time()
    fps_actual = 1 / (current_time - prev_time)
    prev_time = current_time
    
    cv2.putText(frame, f"FPS Actual: {fps_actual:.2f}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    cv2.imshow("Webcam Feed", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Camera initialized with resolution: 640.0x480.0
Camera initialized with resolution: 640x640


In [3]:
import torch
# First, unset any CUDA variables
# os.environ.pop('CUDA_VISIBLE_DEVICES', None)
torch.cuda.empty_cache()

# Then wait a moment and reset
import time
time.sleep(3)

# Now set the device
# os.environ['CUDA_VISIBLE_DEVICES'] = '0'

# Force PyTorch to reinitialize CUDA
torch.cuda.init()

# Verify status
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA device count: {torch.cuda.device_count()}")
print(f"Current CUDA device: {torch.cuda.current_device() if torch.cuda.is_available() else 'None'}")
print(f"CUDA_VISIBLE_DEVICES: {os.environ.get('CUDA_VISIBLE_DEVICES')}")
print(f"CUDA version: {torch.version.cuda}")


CUDA available: True
CUDA device count: 1
Current CUDA device: 0
CUDA_VISIBLE_DEVICES: None
CUDA version: 12.4


In [5]:
# Load your fine-tuned models
# model_8s = YOLO('./runs/detect/hand_detection_8s/weights/best.pt')
model_11s = YOLO('./runs/detect/hand_detection_11s/weights/best.pt')
# model_11n = YOLO('./runs/detect/hand_detection_11n/weights/best.pt')


def run_detection(model, model_name="Model"):
    # Initialize webcam
    cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)  # Use 0 for default webcam
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 640)
    
    prev_time = time.time()
    
    last_result = 0
    
    while True:
        try:
            # Read frame
            ret, frame = cap.read()
            if not ret:
                print("Failed to grab frame")
                break
                
            # removing mirror effect
            frame = frame[:, ::-1, :]

            # run detection
            result = model(frame, conf=0.5, verbose=False)
            last_result = result
                 
            # Draw results on frame
            annotated_frame = result[0].plot()
            
            curr_time = time.time()
            fps_actual = 1 / (curr_time - prev_time)
            prev_time = curr_time
            
            # Add model name to frame
            cv2.putText(annotated_frame, model_name, (10, 30), 
                       cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

            # Add FPS to frame
            cv2.putText(annotated_frame, f"FPS: {fps_actual:.2f}", (10, 60), 
                       cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
            
            # Display frame
            cv2.imshow("Detection", annotated_frame)
            
        except Exception as e:
            print(f"An error occurred: {str(e)}")
            break
        
        # Break loop on 'q' press
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
        
        # cv2.waitKey(0)            
    
    # Cleanup
    cap.release()
    cv2.destroyAllWindows()
    
    print(last_result.boxes)
    

# # Test YOLOv8s
# print("Testing YOLOv8s model... Press 'q' to switch to YOLO11s")
# run_detection(model_8s, "YOLOv8s")

# Test YOLO11s
print("Testing YOLO11s model... Press 'q' to exit")
run_detection(model_11s, "YOLO11s")

Testing YOLO11s model... Press 'q' to exit


AttributeError: 'list' object has no attribute 'boxes'

: 

In [21]:
cap.release()
cv2.destroyAllWindows()

# Citations

```bibtex
@software{yolo11_ultralytics,
  author = {Glenn Jocher and Jing Qiu},
  title = {Ultralytics YOLO11},
  version = {11.0.0},
  year = {2024},
  url = {https://github.com/ultralytics/ultralytics},
  orcid = {0000-0001-5950-6979, 0000-0002-7603-6750, 0000-0003-3783-7069},
  license = {AGPL-3.0}
}
```

```bibtex
@InProceedings{Bambach_2015_ICCV,
author = {Bambach, Sven and Lee, Stefan and Crandall, David J. and Yu, Chen},
title = {Lending A Hand: Detecting Hands and Recognizing Activities in Complex Egocentric Interactions},
booktitle = {The IEEE International Conference on Computer Vision (ICCV)},
month = {December},
year = {2015}
}

```bibtex
@InProceedings{Mittal_2011_BMVC,
author = {Mittal, A. and Zisserman, A. and Torr, P. H. S.},
title = {Hand detection using multiple proposals},
booktitle = {British Machine Vision Conference (BMVC)},
month = {September},
year = {2011}
}
```

```